In [3]:
# Importing the common file shared between both problems
from common import *
from torch.utils.data import DataLoader
from torchvision.transforms import transforms
from transformers import SwinForImageClassification, AutoImageProcessor

## Hyperparameters

In [4]:
# Given in problem prompt
num_classes = 100
batch_size = 32
epochs = 5
learning_rate = 2e-5

# Standard embedding dimension for these types of models
embed_dim = 96

# Given from pretrained swin models tiny and small
image_size = 224
windows_size = 7
patch_size = 4

# Defining input size for FLOPs count
input_size = (1, 3, image_size, image_size)

# Initializing results variable
results = []

## Class Definitions

In [5]:
def window_partition(x, window_size, H, W):
    B, _, C = x.shape
    x = x.reshape(B, H, W, C) # [B, H, W, C]
    x = x.reshape(B, H // window_size, window_size, W // window_size, window_size, C) # [B, Window Row, Row in Window, Window Col, Col in Window, C]
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().reshape(-1, window_size * window_size, C) # Reordering into [B, (Window Row & Col), (Pos in Window), C]

    return windows

In [6]:
def window_reverse(windows, window_size, H, W, B):
    C = windows.shape[-1]
    x = windows.reshape(B, H // window_size, window_size, W // window_size, window_size, C)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().reshape(B, H, W, C)

    return x.reshape(B, H * W, C)

In [7]:
class swinPatchEmbedding(nn.Module):
    def __init__(self, rgb_channels=3):
        super().__init__()

        self.proj = nn.Conv2d(rgb_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)

        return x, H, W

In [8]:
class windowAttention(nn.Module):
    def __init__(self, dim, num_heads, dropout=0.1):
        super().__init__()

        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=dropout, batch_first=True)

    def forward(self, x):
        out, _ = self.attn(x, x, x)
        
        return out

In [9]:
class swinBlock(nn.Module):
    def __init__(self, dim, num_heads, window_size, dropout=0.1):
        super().__init__()

        self.window_size = window_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = windowAttention(dim=dim, num_heads=num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim*4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x, H, W):
        B, _, C = x.shape
        shortcut = x

        x = self.norm1(x)

        windows = window_partition(x, self.window_size, H, W)
        attn_windows = self.attn(windows)
        x = window_reverse(attn_windows, self.window_size, H, W, B)

        x = shortcut + x
        x = x + self.mlp(self.norm2(x))

        return x

In [10]:
class patchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.reduction = nn.Linear((4 * dim), (2 * dim))
        self.norm = nn.LayerNorm(4 * dim)

    def forward(self, x, H, W):
        B, N, C = x.shape

        x = x.reshape(B, H, W, C)
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]

        x = torch.cat([x0, x1, x2, x3], dim=1)
        x = x.reshape(B, -1, 4 * C)
        x = self.norm(x)
        x = self.reduction(x)

        return x, H // 2, W // 2

In [11]:
class swinTransformer(nn.Module):
    def __init__(self, rbg_channels=3, depths=(2, 2, 2), num_head=(3, 6, 12), windows_size=7, dropout=0.1):
        super().__init__()

        self.patch_embed = swinPatchEmbedding(rbg_channels)
        self.stages = nn.ModuleList()
        self.merges = nn.ModuleList()
        
        dim = embed_dim

        for i, depth in enumerate(depths):
            blocks = nn.ModuleList(
                [swinBlock(dim=dim, num_heads=num_head[i], window_size=windows_size, dropout=dropout) for _ in range(depth)])
            
            self.stages.append(blocks)
            if i < (len(depths) - 1):
                self.merges.append(patchMerging(dim))
                dim *= 2

        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)
    
    def forward(self, x):
        x, H, W = self.patch_embed(x)

        for i, blocks in enumerate(self.stages):
            for block in blocks:
                x = block(x, H, W)
            if i < len(self.merges):
                x, H, W = self.merges[i](x, H, W)
        x = self.norm(x)
        x = x.mean(dim=1)

        return self.head(x)

## Helper Functions

In [12]:
def swinLoadCIFAR100(model_name):
    processor = AutoImageProcessor.from_pretrained(model_name)
    
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
    ])

    train_ds = torchvision.datasets.CIFAR100(root='./data', train=True, transform=transform, download=True)
    test_ds = torchvision.datasets.CIFAR100(root='./data', train=False, transform=transform, download=True)

    train_loader = DataLoader(dataset=train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(dataset=test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

In [13]:
def build_pretrained_swin(model_name):
    model = SwinForImageClassification.from_pretrained(model_name, num_labels=num_classes, ignore_mismatched_sizes=True)

    for params in model.swin.parameters():
        params.requires_grad = False
    
    for params in model.classifier.parameters():
        params.requires_grad = True

    return model

In [14]:
def train_swin(model, train_loader, test_loader, lr):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.classifier.parameters(),lr=lr)
    schedule = optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=epochs)

    epoch_time = 0.0

    for epoch in range(epochs):
        start_time = time.time()

        # Training loop
        model.train()
        train_loss = 0.0

        progress_bar = tqdm(train_loader, desc=f'Epoch ({epoch+1}/{epochs})')

        for images, labels in progress_bar:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images).logits
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
        
        schedule.step()

        avg_loss = train_loss / len(train_loader)
        train_time = time.time() - start_time
        epoch_time += train_time

        print(f'Epoch: {epoch+1}    |    Loss: {avg_loss:.4f}    |    Time: {train_time:.3f} sec')
    
    avg_epoch_time = epoch_time / epochs
    test_acc = evaluate(model=model, test_loader=test_loader, device=device)
    num_params = count_params(model=model)

    return {
        "test_acc": test_acc,
        "avg_epoch_time": avg_epoch_time,
        "num_params": num_params
    }

In [15]:
def print_results(results):
    print('Printing Swin Model Comparison Results')
    print('=' * 70)
    for r in results:
        print("\n")
        print(f"Model: {r['name']}")
        print('-' * 80)
        for metric_name, key, fmt in [
            ("Test Accuracy (%)", "test_acc", ".4f"),
            ("No. of Parameters", "num_params", ".4f"),
            ("FLOPs", "flops", ".4f"),
            ("Train Time (s)", "avg_epoch_time", ".3f"),
        ]:
            value = r.get(key)
            try:
                print(f'{metric_name}: {value:{fmt}}')
            except (ValueError, TypeError):
                print(f'{metric_name}: {value}')

## Main Function

### Swin-Tiny Training

In [14]:
print(f'Using device: {device}\n')

print("=" * 80)
print(f'Training Swin-Tiny (pretrained, frozen backbone)')
print("=" * 80)

train_loader_tiny, test_loader_tiny = swinLoadCIFAR100(model_name="microsoft/swin-tiny-patch4-window7-224")
model_tiny = build_pretrained_swin(model_name="microsoft/swin-tiny-patch4-window7-224").to(device)
metrics_tiny = train_swin(model=model_tiny, train_loader=train_loader_tiny, test_loader=test_loader_tiny, lr=learning_rate)
metrics_tiny['flops'] = get_flops(model=model_tiny, input_size=input_size)
metrics_tiny['name'] = 'Swin-Tiny (pretrained, frozen backbone)'

results.append(metrics_tiny)

Using device: cuda

Training Swin-Tiny (pretrained, frozen backbone)


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
Epoch (1/5): 100%|██████████| 1563/1563 [03:25<00:00,  7.61it/s]


Epoch: 1    |    Loss: 4.0548    |    Time: 205.406 sec


...

Epoch: 3    |    Loss: 2.5020    |    Time: 182.960 sec


Epoch (4/5): 100%|██████████| 1563/1563 [03:02<00:00,  8.54it/s]


Epoch: 4    |    Loss: 2.2163    |    Time: 182.925 sec


Epoch (5/5): 100%|██████████| 1563/1563 [03:03<00:00,  8.53it/s]


Epoch: 5    |    Loss: 2.1114    |    Time: 183.242 sec


### Swin-Small Training

In [15]:
print(f'Using device: {device}\n')

print("=" * 80)
print(f'Training Swin-Small (pretrained, frozen backbone)')
print("=" * 80)

train_loader_small, test_loader_small = swinLoadCIFAR100(model_name="microsoft/swin-small-patch4-window7-224")
model_small = build_pretrained_swin(model_name="microsoft/swin-small-patch4-window7-224").to(device)
metrics_small = train_swin(model=model_small, train_loader=train_loader_small, test_loader=test_loader_small, lr=learning_rate)
metrics_small['flops'] = get_flops(model=model_small, input_size=input_size)
metrics_small['name'] = 'Swin-Small (pretrained, frozen backbone)'

results.append(metrics_small)

Using device: cuda

Training Swin-Small (pretrained, frozen backbone)


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/425 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-small-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
Epoch (1/5): 100%|██████████| 1563/1563 [04:57<00:00,  5.25it/s]


Epoch: 1    |    Loss: 3.9632    |    Time: 297.698 sec


...

Epoch: 3    |    Loss: 2.2779    |    Time: 297.674 sec


Epoch (4/5): 100%|██████████| 1563/1563 [04:58<00:00,  5.23it/s]


Epoch: 4    |    Loss: 1.9865    |    Time: 298.840 sec


Epoch (5/5): 100%|██████████| 1563/1563 [04:58<00:00,  5.24it/s]


Epoch: 5    |    Loss: 1.8808    |    Time: 298.491 sec


### Swin-Scratch Training

In [16]:
print(f'Using device: {device}\n')

print("=" * 80)
print(f'Training Swin-Scratch')
print("=" * 80)

train_loader_scratch, test_loader_scratch = load_cifar100(batch_size=batch_size, image_size=image_size)
model_scratch = swinTransformer().to(device)
metrics_scratch = train_model(model=model_scratch, train_loader=train_loader_scratch, test_loader=test_loader_scratch, lr=0.001, epochs=epochs)
metrics_scratch['flops'] = get_flops(model=model_scratch, input_size=input_size)
metrics_scratch['name'] = 'Scratch Swin Transformer Model'

results.append(metrics_scratch)

Using device: cuda

Training Swin-Scratch


Epoch (1/5): 100%|██████████| 1563/1563 [05:52<00:00,  4.43it/s]


Epoch: 1    |    Loss: 4.0643    |    Time: 352.673 sec


Epoch (2/5): 100%|██████████| 1563/1563 [05:53<00:00,  4.42it/s]


Epoch: 2    |    Loss: 3.6744    |    Time: 353.636 sec


...

Epoch: 3    |    Loss: 3.4840    |    Time: 352.959 sec


Epoch (4/5): 100%|██████████| 1563/1563 [05:52<00:00,  4.43it/s]


Epoch: 4    |    Loss: 3.2967    |    Time: 352.832 sec


Epoch (5/5): 100%|██████████| 1563/1563 [05:52<00:00,  4.43it/s]


Epoch: 5    |    Loss: 3.1446    |    Time: 352.594 sec


## Printing Results

In [17]:
print_results(results=results)

Printing Swin Model Comparison Results
Model: Swin-Tiny (pretrained, frozen backbone)
--------------------------------------------------------------------------------
Test Accuracy (%): 62.7700
No. of Parameters: 27596254.0000
FLOPs: 62460334.0000
Train Time (s): 190.178
Model: Swin-Small (pretrained, frozen backbone)
--------------------------------------------------------------------------------
Test Accuracy (%): 67.0400
No. of Parameters: 48914158.0000
FLOPs: 105334894.0000
Train Time (s): 297.921
Model: Scratch Swin Transformer Model
--------------------------------------------------------------------------------
Test Accuracy (%): 22.5100
No. of Parameters: 5077828.0000
FLOPs: 18271204.0000
Train Time (s): 352.939
